# MediEvil text generator

This notebook trains a trigram Markov model on `corpus.csv` and translates a prompt into MediEvil-style text. Run the cells in order to build the model and then generate new sentences from your own prompts.

In [ ]:
import csv
import os
import random
import re
from collections import Counter, defaultdict
from typing import Dict, Iterable, List, Sequence, Tuple

In [ ]:

Token = str
Bigram = Tuple[Token, Token]


def load_corpus(csv_path: str) -> List[str]:
    """Load the text column from the corpus CSV file."""
    texts: List[str] = []
    with open(csv_path, newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if "Corpus" not in reader.fieldnames:
            raise ValueError("Expected a 'Corpus' column in the CSV file")
        for row in reader:
            text = row.get("Corpus")
            if text:
                texts.append(text.strip())
    if not texts:
        raise ValueError("No rows with text were found in the corpus")
    return texts


def tokenize(text: str) -> List[Token]:
    """Tokenize text into words and punctuation, preserving order."""
    return re.findall(r"[A-Za-z']+|[.!?,;:]", text)


def split_sentences(text: str) -> List[str]:
    """Split corpus rows into sentences so starts are better modeled."""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if s.strip()]


class MarkovModel:
    """A simple trigram Markov model for word-level text generation."""

    def __init__(self) -> None:
        self.transitions: Dict[Bigram, Counter[str]] = defaultdict(Counter)
        self.starts: Counter[Bigram] = Counter()
        self.token_counts: Counter[str] = Counter()

    def update(self, tokens: Sequence[Token]) -> None:
        tokens = list(tokens)
        if len(tokens) < 1:
            return
        for token in tokens:
            self.token_counts[token.lower()] += 1
        padded = ["<s>", "<s>", *tokens, "</s>"]
        if len(padded) >= 3:
            self.starts[(padded[0], padded[1])] += 1
        for a, b, c in zip(padded, padded[1:], padded[2:]):
            self.transitions[(a, b)][c] += 1

    def next_token(self, state: Bigram, temperature: float = 1.0) -> Token:
        choices = self.transitions.get(state)
        if not choices:
            return random.choice(list(self.token_counts.keys()))
        tokens, weights = zip(*choices.items())
        if temperature != 1.0:
            weights = [w ** (1.0 / temperature) for w in weights]
        return random.choices(tokens, weights=weights, k=1)[0]

    def generate(
        self,
        seed: Sequence[Token],
        length: int,
        *,
        temperature: float = 0.9,
        stop_on_end: bool = True,
    ) -> str:
        if length < 3:
            raise ValueError("Length should be at least 3 tokens to use a trigram model")
        state = self._seed_state(seed)
        generated: List[Token] = [tok for tok in state if tok not in {"<s>"}]
        while len(generated) < length:
            next_tok = self.next_token(state, temperature=temperature)
            if next_tok == "</s>":
                if stop_on_end and len(generated) >= 6:
                    break
                # If we do not stop here, restart from a sentence open.
                state = self._seed_state([])
                continue
            if next_tok != "<s>":
                generated.append(next_tok)
            state = (state[1], next_tok)
        return self._detokenize(generated)

    def _seed_state(self, seed: Sequence[Token]) -> Bigram:
        if len(seed) >= 2:
            candidate = (seed[-2], seed[-1])
            if candidate in self.transitions:
                return candidate
        if self.starts:
            tokens, weights = zip(*self.starts.items())
            return random.choices(tokens, weights=weights, k=1)[0]
        return random.choice(list(self.transitions.keys()))

    @staticmethod
    def _detokenize(tokens: Iterable[Token]) -> str:
        output: List[str] = []
        for i, token in enumerate(tokens):
            if i == 0:
                output.append(token.capitalize())
                continue
            if re.fullmatch(r"[.!?,;:]", token):
                output[-1] += token
            else:
                output.append(f" {token}")
        return "".join(output)


def build_model(texts: Iterable[str]) -> MarkovModel:
    model = MarkovModel()
    for text in texts:
        for sentence in split_sentences(text):
            tokens = tokenize(sentence)
            model.update(tokens)
    return model


def analyze_corpus(model: MarkovModel) -> str:
    most_common = model.token_counts.most_common(10)
    vocab_size = len(model.token_counts)
    start_count = sum(model.starts.values())
    lines = [
        f"Unique tokens: {vocab_size}",
        f"Number of observed sentence openings: {start_count}",
        "Most common tokens:",
    ]
    for token, count in most_common:
        lines.append(f"  {token}: {count}")
    return "
".join(lines)


## Load the corpus and build the model

Update `CORPUS_PATH` if you want to train on a different CSV file with a `Corpus` column. Running the cell will build the trigram model from the corpus.

In [ ]:
from pathlib import Path

notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
CORPUS_PATH = notebook_dir / 'corpus.csv'

corpus_texts = load_corpus(str(CORPUS_PATH))
model = build_model(corpus_texts)
print(f"Loaded {len(corpus_texts)} corpus rows and {len(model.token_counts)} unique tokens.")


In [ ]:

def translate(
    prompt: str,
    *,
    model: MarkovModel,
    length: int = 40,
    temperature: float = 0.9,
    stop_on_end: bool = True,
    show_analysis: bool = False,
) -> str:
    """Generate MediEvil-style text from the given prompt."""
    if show_analysis:
        print(analyze_corpus(model))
        print("
---
")
    seed_tokens = tokenize(prompt)
    return model.generate(seed_tokens, length, temperature=temperature, stop_on_end=stop_on_end)


## Generate your own translation

Set `example_prompt` to your sentence, adjust `length` as needed, and run the cell to produce MediEvil-styled text. Toggle `show_analysis` to review the corpus statistics before generation.

In [ ]:
example_prompt = "Beware the darkness beyond the hills"
generated_text = translate(example_prompt, model=model, length=30, show_analysis=True)
print(generated_text)
